# Cost & Latency Lab

Run the **same prompt** across four models — `gpt-5.5`, `gpt-5.5-instant`, `gpt-5.4-mini`, `gpt-5.4-nano` — at **three `reasoning_effort` levels** (`none`, `low`, `medium`). Collect **token counts** and **time-to-first-token (TTFT)**, then plot the trade-offs with matplotlib.

This makes the cost/latency/quality triangle concrete: pricier reasoning buys quality but costs tokens and latency.

> **TODO — `gpt-5.5-instant` [unverified]:** if `gpt-5.5-instant` is not yet an API model name, drop it from `MODELS` below (or alias it to `gpt-5.5`).
>
> **Cost note:** this sweep is 4 models × 3 efforts = **12 calls per prompt**. Use a short prompt and run sparingly. Execution is deferred — needs a valid API key.

## Setup

In [ ]:
import os, getpass, time
def _set_env(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")
_set_env("OPENAI_API_KEY")
# !pip install -U openai pandas matplotlib

from openai import OpenAI
import pandas as pd
client = OpenAI()

## 1. Define the sweep

In [ ]:
MODELS = [
    "gpt-5.5",
    "gpt-5.5-instant",   # [unverified] -- remove if not an API name yet
    "gpt-5.4-mini",
    "gpt-5.4-nano",
]
EFFORTS = ["none", "low", "medium"]

PROMPT = "In 4 bullets, explain when to use a message queue vs a database for inter-service communication."

# Per-1M-token output prices (standard tier) for a rough cost estimate.
OUTPUT_PRICE_PER_1M = {
    "gpt-5.5": 30.0,
    "gpt-5.5-instant": 30.0,
    "gpt-5.4-mini": 4.5,
    "gpt-5.4-nano": 1.25,
}

## 2. Run the sweep and measure TTFT + tokens

We stream each call so we can capture **time-to-first-token** (wall-clock from request to first streamed chunk), then read final token usage.

In [ ]:
def measure(model, effort, prompt):
    """Return (ttft_seconds, output_tokens, total_seconds) for one streamed call."""
    start = time.perf_counter()
    ttft = None
    last = None
    stream = client.responses.create(
        model=model,
        input=prompt,
        reasoning={"effort": effort},
        stream=True,
    )
    for event in stream:
        if ttft is None:
            ttft = time.perf_counter() - start   # first chunk arrived
        last = event
    total = time.perf_counter() - start
    # Final usage is on the completed event; fall back gracefully.
    out_tokens = None
    usage = getattr(last, "response", None)
    if usage is not None and getattr(usage, "usage", None):
        out_tokens = usage.usage.output_tokens
    return ttft, out_tokens, total


rows = []
for model in MODELS:
    for effort in EFFORTS:
        try:
            ttft, out_tokens, total = measure(model, effort, PROMPT)
            cost = (out_tokens or 0) / 1_000_000 * OUTPUT_PRICE_PER_1M.get(model, 0)
            rows.append({
                "model": model, "effort": effort,
                "ttft_s": round(ttft or 0, 3),
                "output_tokens": out_tokens,
                "total_s": round(total, 3),
                "est_output_cost_usd": round(cost, 6),
            })
        except Exception as e:
            rows.append({"model": model, "effort": effort, "ttft_s": None,
                         "output_tokens": None, "total_s": None,
                         "est_output_cost_usd": None, "error": str(e)})

df = pd.DataFrame(rows)
df

## 3. Plot the trade-offs

In [ ]:
import matplotlib.pyplot as plt

plot_df = df.dropna(subset=["ttft_s", "output_tokens"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) TTFT by model, grouped by effort
for effort in EFFORTS:
    sub = plot_df[plot_df["effort"] == effort]
    axes[0].plot(sub["model"], sub["ttft_s"], marker="o", label=f"effort={effort}")
axes[0].set_title("Time to First Token by Model")
axes[0].set_ylabel("TTFT (s)")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# (b) Output tokens by model, grouped by effort
for effort in EFFORTS:
    sub = plot_df[plot_df["effort"] == effort]
    axes[1].plot(sub["model"], sub["output_tokens"], marker="s", label=f"effort={effort}")
axes[1].set_title("Output Tokens by Model")
axes[1].set_ylabel("Output tokens")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Reading the results

- **Higher effort → more output/reasoning tokens and higher TTFT.** `none` is fastest; `medium` is the balanced default for `gpt-5.5`.
- **Smaller models (`gpt-5.4-mini`, `gpt-5.4-nano`) are dramatically cheaper** per output token — ideal for high-volume or classification work.
- Pick the **cheapest model + lowest effort** that still passes your quality bar (use the `mini-eval.ipynb` harness to check quality).